# LLMPreSS · Interactive semantic search

Find passages in **ClimbMix** that are semantically relevant to your question—even when they use different words.

Run the cells in order, then use the search box to explore the corpus. The default edition is **compact**. The index and encoder download automatically on first use.

[Project on GitHub](https://github.com/my-unknown-account/LLMPreSS) · [Index collection on Hugging Face](https://huggingface.co/collections/JamshidJDMY/llmpress)

CPU works by default. For GPU query encoding, select **Runtime → Change runtime type → GPU** before running the cells. Vector search uses the CPU.

### Install

Install LLMPreSS and the interactive widgets. Package dependencies are installed automatically. If upgrading an existing installation, use **Runtime → Restart session** after installation, then continue with the index setup cells.

In [ ]:
%pip install -q "git+https://github.com/my-unknown-account/LLMPreSS.git" ipywidgets

## 1. Load the index

Choose the corpus, edition, and local directory below. This cell downloads the index if necessary and loads the encoder once. Reuse the loaded index for successive searches. Passage text is retrieved from the HTTP API when you search.

In [ ]:
from pathlib import Path
import torch
from llmpress import SemanticIndex

corpus = "ClimbMix"
edition = "compact"
index_dir = Path("/content/llmpress-index") if Path("/content").exists() else Path("llmpress-index")
device = "cuda:0" if torch.cuda.is_available() else "cpu"

print(f"Query encoder device: {device}")
index = SemanticIndex(
    corpus,
    edition=edition,
    index_dir=index_dir,
    verbose=True,
    threads=32,
    device=device,
)
print("Ready. Run the next cell to open the search interface.")

## 2. Search by meaning

Enter a question or describe a topic. Retrieve up to **10,000** results (default: **1,000**) and preview the **top 20** passages below. JSON export includes all retrieved results. Results show passage text, similarity scores, and available source metadata. Higher scores indicate stronger matches within the search; they are not probabilities. You can run multiple searches without reloading the index.

In [ ]:
#@title Search passages { display-mode: "form" }
import html
import ipywidgets as widgets
from IPython.display import HTML, display

query_box = widgets.Textarea(
    value="How do plants adapt to drought?",
    placeholder="Ask a question or describe a topic…",
    description="Query:",
    layout=widgets.Layout(width="100%", height="90px"),
)
top_k = widgets.IntSlider(value=1000, min=1, max=10000, description="Results:")
search_button = widgets.Button(description="Search passages", button_style="primary", icon="search")
output = widgets.Output()
results = []
last_query = None

def run_search(_):
    global results, last_query
    query = query_box.value.strip()
    output.clear_output(wait=True)
    with output:
        if not query:
            print("Enter a question or topic first.")
            return
        search_button.disabled = True
        results = []
        try:
            print("Searching…")
            results = index.search(query, top_k=top_k.value)
            last_query = query
            output.clear_output(wait=True)
            if not results:
                print("No passages found.")
            if results:
                print(f"Showing top {min(20, len(results))} of {len(results):,} retrieved passages. Export JSON to save all results.")
            for result in results[:20]:
                title = html.escape(str(result.passage_id or result.numeric_id))
                text = html.escape(result.text or "Passage text unavailable.")
                source = " · ".join(
                    f"{key}: {value}" for key, value in (
                        ("Corpus", result.corpus),
                        ("Source", result.source_rel_path),
                        ("Group", result.source_group),
                        ("Record", result.source_record_index),
                    ) if value is not None
                )
                display(HTML(
                    f'<div style="border:1px solid #94a3b8;border-radius:12px;padding:18px;margin:12px 0">'
                    f'<strong>{result.rank}. {title}</strong> · Score: {result.score:.4f}'
                    f'<p style="white-space:pre-wrap;line-height:1.6">{text}</p>'
                    f'<small>{html.escape(source)}</small></div>'
                ))
        except Exception as error:
            print(f"Search failed: {error}")
        finally:
            search_button.disabled = False

search_button.on_click(run_search)
display(widgets.VBox([query_box, top_k, search_button, output]))

## 3. Export the latest results (optional)

After a successful search, run this cell to save the query and results as JSON. In Colab, the browser will download the file.

In [ ]:
import json
from pathlib import Path

if not results:
    raise ValueError("Run a successful search before exporting.")
export_path = Path("llmpress-results.json")
payload = {
    "query": last_query,
    "results": [result.to_dict() for result in results],
}
export_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
try:
    from google.colab import files
except ImportError:
    print(f"Saved to {export_path.resolve()}")
else:
    files.download(str(export_path))